# Swahili (OpenBible) data prep for OmniVoice finetune — CPU / T4, no A100

**Runtime: ~1.5–2 h** (dominated by streaming-decode + resample of ~20 h of audio and the S3 upload).
No GPU needed. Run top to bottom.

Streams `multilingual-tts/open-bible` (Swahili config; 96 h, single narrator, CC-BY-SA-4.0, parquet),
resamples to 24 kHz, length-filters 2–30 s, holds out an eval set, and uploads wavs + a Yorùbá-style
manifest to S3 so the A100 finetune notebook (nb09) pulls from S3 exactly like the Yorùbá sweep did.
`hf-xet` removed (load-hang gotcha).

## 1. Install (no restart needed — no numpy pin here)

In [ ]:
%pip install --quiet datasets soundfile soxr librosa boto3 "huggingface_hub>=0.24.0" tqdm
%pip uninstall -y hf-xet
print("installs done.")

## 2. Secrets + S3 + constants

In [ ]:
import os, getpass, boto3
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"; os.environ["HF_HUB_ETAG_TIMEOUT"]="60"

BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)

HF_DATASET="multilingual-tts/open-bible"   # CC-BY-SA-4.0, PARQUET (no script builder), 30,634 verses ~96h
HF_CONFIG="Swahili"                          # this repo has per-language configs (Hausa/Igbo/Yoruba too)
LANG_NAME="Swahili"
SR=24000; MIN_DUR,MAX_DUR=2.0,30.0
SUBSET_HOURS=20.0           # enough for the 15 h budget + dev + margin (nb09 takes prefixes)
N_HOLDOUT=12                # held-out eval texts (excluded from train)
SEED=4242

OUT_MANIFEST="tts_data/swahili_gold/openbible_sw.v1.jsonl"
HOLDOUTS_KEY="tts_data/swahili_gold/holdouts.v1.json"
WAV_PREFIX="tts_data/swahili_gold/wav"      # <id>.wav under here
REF_KEY="tts_data/swahili_gold/ref.wav"
WORK="/content/sw_prep" if os.path.isdir("/content") else "/tmp/sw_prep"
LOCAL=os.path.join(WORK,"wav"); os.makedirs(LOCAL,exist_ok=True)
print("S3 OK |",HF_DATASET,"| subset",SUBSET_HOURS,"h | hold-out",N_HOLDOUT)

## 3. Stream → decode → resample 24 kHz → filter → hold-out

Self-introspects the dataset schema (prints the first example's columns) and resolves the text field from
a candidate list, so it survives a column-name difference.

In [ ]:
import soundfile as sf, soxr, numpy as np, random, unicodedata
from datasets import load_dataset, Audio
from tqdm.auto import tqdm

def norm_key(s): return " ".join(unicodedata.normalize("NFC",str(s)).lower().split())

# multilingual-tts/open-bible is a PARQUET dataset (no script builder -> no trust_remote_code / no datasets pin).
# Config = the language ("Swahili"); fields: audio / text / speaker_id / duration_seconds / book / chapter / verse.
ds=load_dataset(HF_DATASET, HF_CONFIG, split="train", streaming=True)
ds=ds.cast_column("audio", Audio(sampling_rate=SR))
# peek schema
_it=iter(ds); _first=next(_it)
print("first-example keys:", list(_first.keys()))
TEXT_CANDS=["text","verse_text","sentence","transcription","transcript","normalized_text","raw_text"]
TEXT_FIELD=next((c for c in TEXT_CANDS if c in _first and isinstance(_first[c],str) and _first[c].strip()), None)
assert TEXT_FIELD, f"no text field found among {TEXT_CANDS}; keys were {list(_first.keys())}"
assert "audio" in _first, "no 'audio' column"
print("using TEXT_FIELD =", TEXT_FIELD)

def _accept(ex):
    a=ex["audio"]; w=np.asarray(a["array"],dtype="float32")
    if a["sampling_rate"]!=SR: w=soxr.resample(w, a["sampling_rate"], SR)
    if w.ndim>1: w=w.mean(-1)
    dur=len(w)/SR
    return w,dur

rows=[]; target=(SUBSET_HOURS+2.0)*3600; acc=0.0
# re-create the stream from scratch so we don't skip _first
ds=load_dataset(HF_DATASET, HF_CONFIG, split="train", streaming=True).cast_column("audio", Audio(sampling_rate=SR))
for i,ex in enumerate(tqdm(ds, desc="stream+resample")):
    txt=ex.get(TEXT_FIELD)
    if not txt or not str(txt).strip(): continue
    w,dur=_accept(ex)
    if dur<MIN_DUR or dur>MAX_DUR: continue
    cid=f"sw_{i:06d}"; loc=os.path.join(LOCAL,cid+".wav")
    sf.write(loc, w, SR)
    rows.append(dict(id=cid,text=str(txt).strip(),duration_sec=round(dur,3),
                     spk=str(ex.get("speaker_id","SPEAKER_00")),local=loc))
    acc+=dur
    if acc>=target: break
from collections import Counter
spkc=Counter(r["spk"] for r in rows); print("speakers:",dict(spkc))
if len(spkc)>1:                       # keep a single voice for clean TTS conditioning
    dom=spkc.most_common(1)[0][0]; rows=[r for r in rows if r["spk"]==dom]
    print(f"-> kept dominant speaker {dom}")
print(f"collected {len(rows)} clips / {sum(r['duration_sec'] for r in rows)/3600:.2f} h")

# hold out N_HOLDOUT texts (3–12 s, for clean eval) -> excluded from train
rng=random.Random(SEED)
eligible=[r for r in rows if 3.0<=r["duration_sec"]<=12.0]
holdout=rng.sample(eligible, min(N_HOLDOUT,len(eligible)))
hold_keys=set(norm_key(r["text"]) for r in holdout)
train=[r for r in rows if norm_key(r["text"]) not in hold_keys]
# reference voice = one clean 3–8 s clip (the single narrator) for cloning/voice-fidelity eval
ref=next((r for r in train if 3.0<=r["duration_sec"]<=8.0), train[0])
print(f"train {len(train)} ({sum(r['duration_sec'] for r in train)/3600:.2f} h) | holdout {len(holdout)} | ref {ref['id']}")

## 4. Upload to S3 (manifest + wavs + holdouts + ref) — size print + confirm gate

In [ ]:
import json, os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
# manifest rows mirror the Yoruba s1_train.v2 schema that nb09 expects
def s3key(r): return f"{WAV_PREFIX}/{r['id']}.wav"
man_rows=[dict(id=r["id"],text=r["text"],duration_sec=r["duration_sec"],
              audio_s3_key=s3key(r),source="openbible_sw",speaker_id=r.get("spk","SPEAKER_00")) for r in train]

total_bytes=sum(os.path.getsize(r["local"]) for r in train)
print(f"about to upload: {len(train)} wavs (~{total_bytes/1e9:.2f} GB) + manifest + holdouts")
print("manifest ->", OUT_MANIFEST, "| wavs ->", WAV_PREFIX+"/", "| ref ->", REF_KEY)

CONFIRM_UPLOAD = True   # set False to dry-run
if CONFIRM_UPLOAD:
    with ThreadPoolExecutor(max_workers=32) as ex:
        list(tqdm(ex.map(lambda r: s3.upload_file(r["local"],BUCKET,s3key(r)), train),
             total=len(train), desc="upload wavs"))
    body="".join(json.dumps(m,ensure_ascii=False)+"\n" for m in man_rows).encode()
    s3.put_object(Bucket=BUCKET,Key=OUT_MANIFEST,Body=body)
    hold_obj=dict(language="swahili",eval_texts=[dict(base=r["id"],text=r["text"]) for r in holdout])
    s3.put_object(Bucket=BUCKET,Key=HOLDOUTS_KEY,Body=json.dumps(hold_obj,ensure_ascii=False,indent=2).encode())
    s3.upload_file(ref["local"],BUCKET,REF_KEY)
    s3.put_object(Bucket=BUCKET,Key=REF_KEY+".json",
                  Body=json.dumps(dict(id=ref["id"],text=ref["text"],dur=ref["duration_sec"]),ensure_ascii=False).encode())
    print("UPLOADED.")
    print(" manifest  -> s3://%s/%s (%d rows)"%(BUCKET,OUT_MANIFEST,len(man_rows)))
    print(" holdouts  -> s3://%s/%s (%d eval texts)"%(BUCKET,HOLDOUTS_KEY,len(holdout)))
    print(" ref wav   -> s3://%s/%s (%s)"%(BUCKET,REF_KEY,ref["id"]))
else:
    print("DRY RUN — set CONFIRM_UPLOAD=True to push.")

### Done → run nb09 (A100)
The finetune notebook reads `tts_data/swahili_gold/openbible_sw.v1.jsonl` + `holdouts.v1.json` + `ref.wav`,
takes hour-budget prefixes of one shuffle, finetunes OmniVoice, and evals CER (Whisper-sw) + voice-fidelity
(SSIM) + listen.